In [1]:
import glob
import yaml
import ml_collections
from tqdm import tqdm
from utils.utils import *
from utils.metric import *
from utils.eval_portfolio.summary import *
from utils.eval_portfolio.strategies import log_return_to_price
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath
from models.DMS_TCN import DMSTCN
from torch.utils.data import DataLoader, TensorDataset
from modules.diffusion import TimeSeriesGaussianDiffusion, generate_cosine_schedule


file_manager = quantDiffusionPath()
gc.collect()

0

# Preprocessing

In [2]:
def fit_transform_garch_12d(df_log_returns, df_rv, rv_window=10, min_periods=5):
    eps = 1e-8
    
    # --- key fix: align indices ---
    # keep only the common index of the two dataframes
    common_idx = df_log_returns.index.intersection(df_rv.index)
    df_log_returns = df_log_returns.loc[common_idx]
    df_rv = df_rv.loc[common_idx]

    T, C = df_log_returns.shape
    print(f"Synced Shape - Returns: {df_log_returns.shape}, RV: {df_rv.shape}")

    # 1. forward-window realized volatility
    future_rv = np.sqrt(
        df_rv.rolling(rv_window, min_periods=min_periods)
        .mean()
        .shift(-(rv_window - 1))
    )

    # 2. decompose into bulk(Z) and vol(V)
    # indices aligned, division is now safe
    Z_t = (df_log_returns / (future_rv + eps))
    V_t = np.log(future_rv + eps)
    
    # 3. drop NaNs 
    # (drop NaNs from pct_change and the forward-window tail)
    combined = pd.concat([Z_t, V_t], axis=1).dropna()
    
    # re-check dimensions
    Z_t_clean = combined.iloc[:, :C]
    V_t_clean = combined.iloc[:, C:]
    print(f"Final Cleaned Shape: {combined.shape}")

    # 4. combine to 12d and scale
    scaled_12d_raw = combined.values
    scaled_12d, scalers = scaling(scaled_12d_raw)
    
    # 5. orthogonality score
    corrs = []
    for i in range(C):
        c = np.corrcoef(Z_t_clean.iloc[:, i], V_t_clean.iloc[:, i])[0, 1]
        corrs.append(c)
    
    avg_orthogonality = np.mean(np.abs(corrs))
    print(f"Average Orthogonality Score (Corr): {avg_orthogonality:.4f}")
    scaled_12d_raw = pd.concat([Z_t, np.log(future_rv)], axis=1).dropna().values

    return scaled_12d_raw, scaled_12d, scalers


def inverse_garch_12d_to_6d(generated_12d, scalers):
    """
    Reconstruct the original 6d returns from the 12d generated data.

    Args:
        generated_12d: (B, 12, T) or (B, T, 12) shaped tensor/array
        scalers_Z: list of residual(Z) scalers fit on train
        scalers_V: list of log-vol(V) scalers fit on train
    Returns:
        X_reconstructed: reconstructed (B, 6, T) or (B, T, 6) original 6d tensor
    """
    # 3. split 12d (first 6: residual Z, last 6: log-vol V)
    inv_12d = inverse_scaling(generated_12d.transpose(1, 2), scalers)

    Z_inv = inv_12d[:, :6, :]
    V_inv = inv_12d[:, 6:, :]

    # 5. recover volatility (undo log)
    # guard: clip before exp so early-training extremes do not overflow (cap ~10)
    V_inv = np.clip(V_inv, a_min=-20.0, a_max=10.0)
    sigma_t = np.exp(V_inv)

    # 6. final 6d synthesis (Z * sigma)
    # multiply calm Z noise by vol spikes to reproduce real crashes/rallies
    X_reconstructed = Z_inv * sigma_t

    return X_reconstructed

In [3]:
# --- [0. setup] ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 256

df = pd.read_csv(f"./data/indices.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df.set_index("Date", inplace=True)
df = df.apply(pd.to_numeric).astype(float)

# ===== [winning config] RV=Close_to_Close, rv_window=10 =====
# best overall in the RV sweep (20 configs): 2nd EMD + 2nd mean TailGAN RE VaR/ES (overall 1st),
# lowest RE_ES(0.01) (0.3408).
rv = pd.read_csv(file_manager.get_rv_file("Close_to_Close"), index_col=0)
rv.index = pd.to_datetime(rv.index)
rv.index.name = "Date"


# same RV floor as the sweep: clip neg/zero to per-asset positive 1% quantile (no effect for positive Close_to_Close)
def floor_rv(_rv):
    _rv = _rv.copy()
    for _c in _rv.columns:
        _pos = _rv[_c][_rv[_c] > 0]
        _fl = float(_pos.quantile(0.01)) if len(_pos) else 1e-8
        _rv[_c] = _rv[_c].clip(lower=_fl)
    return _rv


rv = floor_rv(rv)

df_log_returns = np.log1p(df.pct_change()).dropna()
log_returns = df_log_returns.values
init_price_raw = np.array(df)[:-1, :]

# --- [1. early split (chronological)] ---
total_len = len(log_returns)
train_size = int(0.7 * total_len)

train_raw = df_log_returns.iloc[:train_size, :]
test_raw = df_log_returns.iloc[train_size:, :]  # test stays as raw 6d

train_init_raw = init_price_raw[:train_size]
test_init_raw = init_price_raw[train_size:]

# --- [2. GARCH 12d preprocessing (Train ONLY)] ---
raw_12d, train_12d, scalers = fit_transform_garch_12d(df_log_returns, rv, rv_window=10)
# scaled row 0 = original series index (FFD burn-in removed)

train_12d = train_12d[:train_size]

# --- [3. windowing] ---
# build 12d windows for train
train_windowed = torch.from_numpy(rolling_window(train_12d, seq_len)).float()
train_init_windowed = (
    torch.from_numpy(train_init_raw[: -(seq_len - 1)]).float().unsqueeze(1)
)

test_windowed = torch.from_numpy(rolling_window(test_raw.values, seq_len)).float()
test_init_windowed = (
    torch.from_numpy(test_init_raw[: -(seq_len - 1)]).float().unsqueeze(1)
)

# (B, T, C) -> (B, C, T) axis swap
training_data = train_windowed.clone().detach()  # (B, 256, 12)
test_data = test_windowed.clone().detach()  # (B, 256, 6)

training_set = TensorDataset(training_data)
test_set = TensorDataset(test_data)

train_dl = DataLoader(training_set, batch_size=256, shuffle=False)
test_dl = DataLoader(test_set, batch_size=256, shuffle=False)

print("\nData ready.")
print(f"Train DataLoader batch: {next(iter(train_dl))[0].shape}")
print(f"Test DataLoader batch: {next(iter(test_dl))[0].shape}")

c:\Users\Juhwankim\miniconda3\envs\research\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


Synced Shape - Returns: (5976, 6), RV: (5976, 6)
Final Cleaned Shape: (5967, 12)
Average Orthogonality Score (Corr): 0.0852

Data ready.
Train DataLoader batch: torch.Size([256, 256, 12])
Test DataLoader batch: torch.Size([256, 256, 6])


# Ablation Studies

baseline(= RV decomposition(12d) + LambertW scaling + DMS-TCN) : train/validate three variants, each removing/replacing one component, then collect per-model results and a summary table.

| category | change | enc_in | backbone |
|------|--------|--------|----------|
| 1. w/o RV Decomposition | no RV decomposition -> raw 6d | 6 | DMS-TCN |
| 2. w/o LambertW | drop only LambertW(Gaussianize) from scaling | 12 | DMS-TCN |
| 3. w/o DMS-TCN | replace backbone with quantTCN | 12 | quantTCN |
| 4. summary | per-model results + ablation summary CSV | - | - |

> **Prerequisite**: run the Preprocessing cells (0-3) first so that `df_log_returns`, `rv`, `train_size`, `train_raw`, `test_data`, `train_init_raw`, `test_init_raw`, `train_12d`, `scalers` are defined. Run the shared-utils cell once, then proceed 1->2->3->4.

In [4]:
# ===== Ablation shared utils =====
# train/validate/generate/evaluate code is the same as the baseline cells;
# only preprocessing / scaling / backbone are injected per ablation.
# (use after running Preprocessing cells 0-3: df_log_returns, rv, train_size, train_raw,
#  test_data, train_init_raw, test_init_raw, train_12d, scalers needed)
import os
import glob
import pandas as pd
from sklearn.preprocessing import StandardScaler
from models.DMS_TCN import DMSTCN
from models.quantTCN import QuantTCN

NUM_EPOCHS = 45
N_RUNS = 5
VAL_START = 25
TARGET_N = 2000  # number of synthetic sequences to keep (after filtering) per run
CLIP_LATENT = 8.0  # latent clip range before inverse (+/-sigma): avoids LambertW exp overflow/ValueError

ABL = {}  # ablation config registry (filled per section) -> generated/evaluated together in section 4


# ---- (for w/o LambertW) scaling without the Gaussianize (Lambert W) step ----
def scaling_no_lambertw(data):
    scalers, scaled_data = [], []
    for i in range(data.shape[1]):
        feature_data = data[:, i].reshape(-1, 1)
        ss1, ss2 = StandardScaler(), StandardScaler()
        feature_scaled = ss2.fit_transform(ss1.fit_transform(feature_data))
        scalers.append((ss1, ss2, None))
        scaled_data.append(feature_scaled.flatten())
    return np.array(scaled_data).T, scalers


def inverse_scaling_no_lambertw(y, scalers):
    y = y.cpu().detach().numpy() if hasattr(y, "cpu") else y
    y_original = np.zeros_like(y)
    for idx in range(y.shape[1]):
        ss1, ss2, _ = scalers[idx]
        y_feature = y[:, idx, :]
        EPS = 1e-8
        y_feature = (y_feature - y_feature.mean(axis=0, keepdims=True)) / (
            y_feature.std(axis=0, keepdims=True) + EPS
        )
        orig_shape = y_feature.shape
        y_flat = y_feature.reshape(-1, 1)
        y_flat = ss1.inverse_transform(ss2.inverse_transform(y_flat))
        y_original[:, idx, :] = y_flat.reshape(orig_shape)
    return y_original


# ---- 12d -> 6d reconstruction (inverse-scaling injectable; same as inverse_garch_12d_to_6d) ----
def garch_12d_to_6d(generated_12d, scalers, inv_scaling_fn=inverse_scaling):
    inv_12d = inv_scaling_fn(generated_12d.transpose(1, 2), scalers)
    Z_inv = inv_12d[:, :6, :]
    V_inv = np.clip(inv_12d[:, 6:, :], a_min=-20.0, a_max=10.0)
    return Z_inv * np.exp(V_inv)


# ---- (for w/o RV Decomposition) scale the raw 6d log-returns directly ----
def fit_transform_no_rv(df_log_returns, scaling_fn=scaling):
    raw_6d = df_log_returns.dropna().values
    scaled_6d, scalers = scaling_fn(raw_6d)
    return raw_6d, scaled_6d, scalers


# ---- 12d preprocessing (scaling-fn injectable; same as fit_transform_garch_12d) ----
def fit_transform_garch_12d_with(
    df_log_returns, df_rv, scaling_fn=scaling, rv_window=20, min_periods=5
):
    eps = 1e-8
    common_idx = df_log_returns.index.intersection(df_rv.index)
    dlr, drv = df_log_returns.loc[common_idx], df_rv.loc[common_idx]
    future_rv = np.sqrt(
        drv.rolling(rv_window, min_periods=min_periods).mean().shift(-(rv_window - 1))
    )
    Z_t = dlr / (future_rv + eps)
    V_t = np.log(future_rv + eps)
    combined = pd.concat([Z_t, V_t], axis=1).dropna()
    scaled_12d, scalers = scaling_fn(combined.values)
    raw_12d = pd.concat([Z_t, np.log(future_rv)], axis=1).dropna().values
    return raw_12d, scaled_12d, scalers


# ---- backbone / diffusion builder ----
def build_backbone(name, enc_in, max_seq_len):
    if name == "quanttcn":
        return QuantTCN(
            {
                "enc_in": enc_in,
                "hidden_dim": 128,
                "dilations": [1, 2, 4, 8, 16, 32],
                "kernel_size": 2,
                "dropout": 0.05,
                "use_time_emb": True,
                "time_emb_dim": 128,
            }
        )
    return DMSTCN(
        {
            "enc_in": enc_in,
            "dims": [128, 128, 128],
            "large_size": [15, 7, 5],
            "small_size": [5, 5, 5],
            "time_emb_dim": 128,
            "use_time_emb": True,
            "use_cross_attention": False,
            "max_seq_len": max_seq_len,
            "use_sequence_pos_encoding": False,
        }
    )


def build_diffusion(enc_in, backbone, seq_len_):
    ts_model = build_backbone(backbone, enc_in, seq_len_)
    betas = generate_cosine_schedule(75, 1.5, 1e-5, 0.5)
    diffusion = TimeSeriesGaussianDiffusion(
        ts_model,
        seq_len=seq_len_,
        num_channels=enc_in,
        betas=betas,
        loss_type="11",
        ema_decay=0.9999,
        ema_start=2000,
        ema_update_rate=1,
        sampling_type="student_t",
        prediction_type="epsilon",
    ).to(device)
    return ts_model, diffusion


# ---- training (cell 8 + 10) ----
def train_ablation(tag, train_scaled, backbone, num_epochs=NUM_EPOCHS, batch_size=256):
    print(f"\n{'#'*70}\n# TRAIN [{tag}] backbone={backbone}, enc_in={train_scaled.shape[1]}\n{'#'*70}")
    training_data = torch.from_numpy(rolling_window(train_scaled, seq_len)).float()
    enc_in = training_data.shape[2]
    train_dl = DataLoader(
        TensorDataset(training_data.clone().detach()), batch_size=batch_size, shuffle=False
    )
    ts_model, diffusion = build_diffusion(enc_in, backbone, training_data.shape[1])
    optimizer = torch.optim.Adam(diffusion.parameters(), lr=2e-4, weight_decay=1e-5)
    weight_dir = file_manager.get_weight_path()
    os.makedirs(weight_dir, exist_ok=True)
    count_parameters(ts_model)

    clean_memory(device)
    pbar = tqdm(range(1, num_epochs + 1))
    for epoch in pbar:
        diffusion.train()
        epoch_loss, n_batches = 0.0, 0
        for batch in train_dl:
            x = batch[0].to(device)
            loss = diffusion(x)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            diffusion.update_ema()
            epoch_loss += loss.item()
            n_batches += 1
            del x, loss
        pbar.set_description(f"[{tag}] Epoch {epoch}")
        pbar.set_postfix(train_loss=f"{epoch_loss / n_batches:.4f}")
        torch.save(diffusion.state_dict(), f"{weight_dir}/{tag}_{epoch}.pt")
        if epoch % 10 == 0:
            clean_memory(device)
    return diffusion


# ---- validation / best-epoch selection (cell 12) ----
def validate_ablation(tag, diffusion, to_6d_fn, num_epochs=NUM_EPOCHS, val_start=VAL_START):
    weight_dir = file_manager.get_weight_path()
    result_path = file_manager.get_result_path(tag)
    os.makedirs(result_path, exist_ok=True)
    min_dist, best_epoch = float("inf"), -1
    real_data_val = (
        torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2)
    )
    for epoch in tqdm(range(val_start, num_epochs + 1), desc=f"[{tag}] validate"):
        ckpt_path = f"{weight_dir}/{tag}_{epoch}.pt"
        if not os.path.exists(ckpt_path):
            continue
        diffusion.load_state_dict(torch.load(ckpt_path, map_location=device))
        diffusion.eval()
        with torch.no_grad():
            fake = diffusion.sample(batch_size=100, device=device, use_ema=True)
            fake = torch.clamp(fake, -CLIP_LATENT, CLIP_LATENT)
            with np.errstate(over="ignore", invalid="ignore"):
                fake_data = to_6d_fn(fake)
            emd = compute_avg_emd(real_data_val, fake_data, 100)
            if emd < min_dist:
                min_dist, best_epoch = emd, epoch
        del fake, fake_data
        clean_memory(device)

    if best_epoch == -1:
        print(f"[{tag}] no valid checkpoint")
        return -1
    print(f"[{tag}] Best Epoch: {best_epoch} (EMD: {min_dist:.5f})")
    best_ckpt_path = f"{weight_dir}/{tag}_{best_epoch}.pt"
    final_best_path = f"{result_path}/quantDiffusion_best({best_epoch}).pt"
    diffusion.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    torch.save(diffusion.state_dict(), final_best_path)
    for fp in glob.glob(f"{weight_dir}/{tag}_*.pt"):
        if fp != best_ckpt_path:
            try:
                os.remove(fp)
            except Exception:
                pass
    print(f"[{tag}] best model saved: {final_best_path}")
    return best_epoch


# ---- generation (cell 14 + 17): generate until the target count survives ----
def generate_ablation(
    tag, enc_in, backbone, to_6d_fn, n_runs=N_RUNS, target_n=TARGET_N,
    gen_batch=100, max_total=2600,
):
    result_path = file_manager.get_result_path(tag)
    best_files = glob.glob(f"{result_path}/quantDiffusion_best*.pt")
    if not best_files:
        raise FileNotFoundError(f"[{tag}] no best weight ({result_path}) - run training/validation first")
    _, diffusion = build_diffusion(enc_in, backbone, seq_len)
    target_file = max(best_files, key=os.path.getmtime)
    diffusion.load_state_dict(torch.load(target_file, map_location=device))
    diffusion.to(device)
    diffusion.eval()
    print(f"[{tag}] loaded: {os.path.basename(target_file)}")

    real_data = test_data.transpose(1, 2).cpu().detach().numpy()
    train_data_6d = (
        torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2).numpy()
    )
    min_vals = train_data_6d.min(axis=(0, 2), keepdims=True)
    max_vals = train_data_6d.max(axis=(0, 2), keepdims=True)
    range_margin = (max_vals - min_vals) * 0.1
    lower_bound, upper_bound = min_vals - range_margin, max_vals + range_margin

    def _survivors(raw_list):
        # inverse_scaling normalizes with batch stats, so convert all raw at once then filter.
        # LambertW inverse exp overflow -> inf/nan sequences are dropped by isfinite,
        # the while loop below keeps generating until the target is met (auto retry).
        # extreme outliers corrupt the internal standardization in inverse_scaling so LambertW exp
        # would overflow(->inf, sklearn ValueError); clip latent before inverse to prevent it.
        samples = torch.clamp(torch.cat(raw_list, dim=0), -CLIP_LATENT, CLIP_LATENT)
        with np.errstate(over="ignore", invalid="ignore"):
            f6 = to_6d_fn(samples)
        finite = np.isfinite(f6).all(axis=(1, 2))
        m = finite & np.all((f6 >= lower_bound) & (f6 <= upper_bound), axis=(1, 2))
        return f6[m]

    for run_idx in range(n_runs):
        # 1st pass: assume ~60% survival and over-generate vs target
        n_init = int(np.ceil(target_n / 0.6 / gen_batch))
        raw = []
        with torch.no_grad():
            for _ in range(n_init):
                raw.append(diffusion.sample(batch_size=gen_batch, device=device, use_ema=True))
        survivors = _survivors(raw)
        # if short, generate more based on survival rate (until target, capped at max_total)
        while len(survivors) < target_n and len(raw) * gen_batch < max_total:
            rate = max(len(survivors), 1) / (len(raw) * gen_batch)
            need = target_n - len(survivors)
            n_add = int(np.ceil(need / max(rate, 0.2) / gen_batch)) + 1
            with torch.no_grad():
                for _ in range(n_add):
                    raw.append(diffusion.sample(batch_size=gen_batch, device=device, use_ema=True))
            survivors = _survivors(raw)
            print(f"  [{tag}] run {run_idx}: generated {len(raw) * gen_batch} -> survived {len(survivors)} (target {target_n})")
        clean_memory(device)

        if len(survivors) >= target_n:
            idx = np.random.choice(len(survivors), target_n, replace=False)
            final_fake_data = survivors[idx]
        else:
            print(f"[{tag}] run {run_idx}: survived {len(survivors)} < target {target_n} even at cap {max_total}; saving what we have.")
            final_fake_data = survivors
        np.save(os.path.join(result_path, f"{run_idx}.npy"), final_fake_data)
        print(f"[{tag}] Run {run_idx + 1}/{n_runs}: {final_fake_data.shape}")
    clean_memory(device)
    return real_data


# ---- per-model eval: gen quality (cell 19), files named by tag ----
def evaluate_ablation(tag, real_data, n_runs=N_RUNS):
    result_path = file_manager.get_result_path(tag)
    windows = [1, 5, 20, 100]

    total_result_df = pd.DataFrame()
    for seed in range(n_runs):
        fake_data = np.load(f"{result_path}/{seed}.npy")
        result_df = pd.DataFrame()
        result_df = pd.concat([result_df, calculate_ccf(real_data, fake_data, 0)], axis=0)
        total_CCF_df = pd.DataFrame(columns=["Average"])
        for i in range(1, 11):
            total_CCF_df = pd.concat(
                [total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1
            )
        result_df = pd.concat(
            [result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=["Average"])], axis=0
        )
        result_df = pd.concat(
            [result_df, calculate_distribution_scores(real_data, fake_data, windows)], axis=0
        )
        result_df = pd.concat(
            [result_df, calculate_skew_kurtosis(real_data, fake_data, windows)], axis=0
        )
        _, acf_df = calculate_acf_score(real_data, fake_data, 40)
        result_df = pd.concat([result_df, acf_df], axis=0)
        _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
        result_df = pd.concat([result_df, leverage_df], axis=0)
        total_result_df = pd.concat([total_result_df, result_df], axis=1)

    statistics_df = pd.DataFrame(index=total_result_df.index, columns=["MEAN", "STD"])
    statistics_df["MEAN"] = total_result_df.mean(axis=1).values
    statistics_df["STD"] = total_result_df.std(axis=1).values
    statistics_df.to_csv(file_manager.get_result_file(f"individual__{tag}__gen_quality"))

    print(f"[{tag}] per-model result saved: {tag}_gen_quality.csv")


# ---- summary table (baseline + ablations in one table) ----
def _pm(m, s, nd=4):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if (np.isfinite(m) and np.isfinite(s)) else ""


def aggregate_gen_quality(tag_label_map, out_name="ablation__gen_quality"):
    cols = {}
    for tag, label in tag_label_map.items():
        f = file_manager.get_result_file(f"individual__{tag}__gen_quality")
        if not os.path.exists(f):
            print(f"  (skip {tag}: {os.path.basename(f)} missing)")
            continue
        df = pd.read_csv(f, index_col=0)
        cols[label] = pd.Series(
            [_pm(m, s) for m, s in zip(df["MEAN"], df["STD"])], index=df.index
        )
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


# ---- rebuild ABL without training after a kernel restart (for generate/eval from saved weights) ----
def build_abl_registry():
    """re-run only preprocessing/scaling to restore ABL config (enc_in, backbone, to_6d_fn).
    scalers are deterministic given the data, so they reproduce exactly (no retraining needed).
    generate_ablation loads quantDiffusion_best*.pt saved under result_path.
    (needs Preprocessing cells 0-3: df_log_returns, rv, train_12d, scalers)"""
    reg = {}
    reg["baseline"] = dict(
        label="Baseline", enc_in=12, backbone="moderntcn", train_scaled=train_12d,
        to_6d_fn=lambda s, sc=scalers: garch_12d_to_6d(s, sc, inv_scaling_fn=inverse_scaling),
    )
    _, _s6, sc_wo_rv = fit_transform_no_rv(df_log_returns, scaling_fn=scaling)
    reg["wo_rv"] = dict(
        label="w/o RV Decomp.", enc_in=6, backbone="moderntcn", train_scaled=_s6[:train_size],
        to_6d_fn=lambda s, sc=sc_wo_rv: inverse_scaling(s.transpose(1, 2), sc),
    )
    _, _s12lw, sc_wo_lw = fit_transform_garch_12d_with(
        df_log_returns, rv, scaling_fn=scaling_no_lambertw, rv_window=10
    )
    reg["wo_lambertw"] = dict(
        label="w/o LambertW", enc_in=12, backbone="moderntcn", train_scaled=_s12lw[:train_size],
        to_6d_fn=lambda s, sc=sc_wo_lw: garch_12d_to_6d(s, sc, inv_scaling_fn=inverse_scaling_no_lambertw),
    )
    reg["wo_moderntcn"] = dict(
        label="w/o DMS-TCN", enc_in=12, backbone="quanttcn", train_scaled=train_12d,
        to_6d_fn=lambda s, sc=scalers: garch_12d_to_6d(s, sc, inv_scaling_fn=inverse_scaling),
    )
    return reg


# ---- 3-tier: reuse existing npy > regenerate from weight > train then generate ----
def ensure_synthetic(tag, cfg, n_runs=N_RUNS):
    rp = file_manager.get_result_path(tag)
    if len(glob.glob(f"{rp}/[0-9]*.npy")) >= n_runs:
        print(f"  [{tag}] tier-1: existing npy -> skip generation")
        return
    if glob.glob(f"{rp}/quantDiffusion_best*.pt"):
        print(f"  [{tag}] tier-2: weight found -> regenerate")
        generate_ablation(tag, cfg["enc_in"], cfg["backbone"], cfg["to_6d_fn"])
        return
    print(f"  [{tag}] tier-3: no npy/weight -> train then generate")
    diffusion = train_ablation(tag, cfg["train_scaled"], cfg["backbone"])
    validate_ablation(tag, diffusion, cfg["to_6d_fn"])
    generate_ablation(tag, cfg["enc_in"], cfg["backbone"], cfg["to_6d_fn"])


## 0. Baseline (full model) retraining
RV decomp(12d, Close_to_Close/w10) + LambertW scaling + DMS-TCN  here, together with the ablations under the same config (no dependency on a separate modeling notebook).

In [5]:
# ===== 0. Baseline (full model) retraining (moderntcn + full scaling + 12d) =====
# train baseline directly here -> fully controlled comparison with the 3 wo_* under the same config (Close_to_Close/w10).
ABL["baseline"] = dict(
    label="Baseline",
    train_scaled=train_12d,
    enc_in=12,
    backbone="moderntcn",
    to_6d_fn=lambda s: garch_12d_to_6d(s, scalers, inv_scaling_fn=inverse_scaling),
)


## 1. w/o RV Decomposition
Train directly on raw 6d log-returns without RV decomposition (Z/log-vol 12d). Scaling is the same as baseline (incl. LambertW). Generate 6d and inverse-transform directly to 6d.

In [6]:
# train/generate directly on raw 6d log-returns without RV decomposition (scaling same as baseline, incl. LambertW)
_, scaled6_full, scalers_wo_rv = fit_transform_no_rv(df_log_returns, scaling_fn=scaling)
train6 = scaled6_full[:train_size]

ABL["wo_rv"] = dict(
    label="w/o RV Decomp.",
    train_scaled=train6,
    enc_in=6,
    backbone="moderntcn",
    to_6d_fn=lambda s: inverse_scaling(s.transpose(1, 2), scalers_wo_rv),
)


## 2. w/o LambertW
Keep the RV decomposition (12d) same as baseline, and remove only the LambertW(Gaussianize) step from scaling (`SS1 -> SS2`).

In [7]:
# RV decomposition (12d) same as baseline; drop only the LambertW(Gaussianize) step from scaling (SS1 -> SS2)
# winning config : rv_window=10 (rv prepared as Close_to_Close + floor_rv in cell [0])
_, train12_wo_lw_full, scalers_wo_lw = fit_transform_garch_12d_with(
    df_log_returns, rv, scaling_fn=scaling_no_lambertw, rv_window=10
)
train12_wo_lw = train12_wo_lw_full[:train_size]

ABL["wo_lambertw"] = dict(
    label="w/o LambertW",
    train_scaled=train12_wo_lw,
    enc_in=12,
    backbone="moderntcn",
    to_6d_fn=lambda s: garch_12d_to_6d(s, scalers_wo_lw, inv_scaling_fn=inverse_scaling_no_lambertw),
)


## 3. w/o DMS-TCN
Keep preprocessing/scaling same as baseline; replace only the backbone DMS-TCN -> quantTCN.

In [8]:
# replace backbone DMS-TCN -> quantTCN (preprocessing/scaling same as baseline: reuse train_12d, scalers)
ABL["wo_moderntcn"] = dict(
    label="w/o DMS-TCN",
    train_scaled=train_12d,
    enc_in=12,
    backbone="quanttcn",
    to_6d_fn=lambda s: garch_12d_to_6d(s, scalers, inv_scaling_fn=inverse_scaling),
)


## 4. Per-model results + Ablation summary table
Using the best weights selected in 1-3, generate each ablation's synthetic data and per-model eval CSVs, then build the summary table CSV including baseline (`ablation_summary_gen_quality.csv`).

In [9]:
# ===== 4-1. per-model results: reuse synthetic data (.npy) if present, else generate then evaluate =====
# if ABL is empty (e.g., kernel restart), rebuild from saved weights without training.
# (Preprocessing cells 0-3 must have run first: df_log_returns, rv, train_12d, scalers)
if not ABL:
    print("ABL is empty; rebuilding via build_abl_registry() (no training, using saved weights).")
    ABL = build_abl_registry()
    for _t, _c in ABL.items():
        _bf = glob.glob(f"{file_manager.get_result_path(_t)}/quantDiffusion_best*.pt")
        print(f"  - {_t:14s} backbone={_c['backbone']:9s} "
              f"weight={'O ' + os.path.basename(_bf[0]) if _bf else 'X (needs training)'}")

# real_data is independent of generation (test 6d) - computed once
real_data = test_data.transpose(1, 2).cpu().detach().numpy()

for tag, cfg in ABL.items():
    print(f"\n{'='*70}\ngenerate/evaluate per-model: {cfg['label']} ({tag})\n{'='*70}")
    ensure_synthetic(tag, cfg)  # tier1 npy > tier2 weight > tier3 train
    evaluate_ablation(tag, real_data)


generate/evaluate per-model: Baseline (baseline)
  [baseline] tier-1: existing npy -> skip generation
[baseline] per-model result saved: baseline_gen_quality.csv

generate/evaluate per-model: w/o RV Decomp. (wo_rv)
  [wo_rv] tier-1: existing npy -> skip generation
[wo_rv] per-model result saved: wo_rv_gen_quality.csv

generate/evaluate per-model: w/o LambertW (wo_lambertw)
  [wo_lambertw] tier-1: existing npy -> skip generation
[wo_lambertw] per-model result saved: wo_lambertw_gen_quality.csv

generate/evaluate per-model: w/o DMS-TCN (wo_moderntcn)
  [wo_moderntcn] tier-1: existing npy -> skip generation
[wo_moderntcn] per-model result saved: wo_moderntcn_gen_quality.csv


In [10]:
# ===== 4-2. Ablation summary table - works from saved per-model CSVs only =====
# a fixed mapping independent of ABL (memory), so even running only this cell after a kernel restart
# collects {tag}_gen_quality.csv from disk into the summary table.
# (missing tags are skipped automatically.)
tag_label_map = {
    "baseline": "Baseline",
    "wo_rv": "w/o RV Decomp.",
    "wo_lambertw": "w/o LambertW",
    "wo_moderntcn": "w/o DMS-TCN",
}

# check which tags actually have results on disk (X means that ablation's 4-1 was not run -> skip)
print("result availability (X means that ablation needs 4-1 generate/eval):")
for tag, label in tag_label_map.items():
    gq = os.path.exists(file_manager.get_result_file(f"individual__{tag}__gen_quality"))
    print(f"  - {label:20s}: gen_quality={'O' if gq else 'X'}")

print("\n[Ablation summary - Generation Quality]")
summary_gen_quality = aggregate_gen_quality(tag_label_map)
print(summary_gen_quality)


result availability (X means that ablation needs 4-1 generate/eval):
  - Baseline            : gen_quality=O
  - w/o RV Decomp.      : gen_quality=O
  - w/o LambertW        : gen_quality=O
  - w/o DMS-TCN         : gen_quality=O

[Ablation summary - Generation Quality]
saved: c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/statistics/ablation__gen_quality.csv
                        Baseline   w/o RV Decomp.     w/o LambertW  \
CCF(τ = 0)_MAE   0.0572 ± 0.0007  0.0524 ± 0.0005  0.0709 ± 0.0003   
CCF(τ = 0)_FROB  0.4520 ± 0.0054  0.3861 ± 0.0038  0.5918 ± 0.0024   
CCF(τ = 0)_MSE   0.0057 ± 0.0001  0.0042 ± 0.0001  0.0097 ± 0.0001   
CCF(τ > 0)_MAE   0.0341 ± 0.0002  0.0342 ± 0.0001  0.0420 ± 0.0001   
CCF(τ > 0)_FROB  0.6659 ± 0.0039  0.6540 ± 0.0028  0.8527 ± 0.0031   
CCF(τ > 0)_MSE   0.0022 ± 0.0000  0.0021 ± 0.0000  0.0037 ± 0.0000   
EMD(1)           0.0014 ± 0.0000  0.0013 ± 0.0000  0.0014 ± 0.0000   
EMD(5)           0.0033 ± 0.0000  0.0033 ± 0.0001  0.0034 ± 0.0000  

In [11]:
# ===== 4-3. TailGAN-style VaR/ES (ported) - ablation summary (evaluate existing npy, no retraining/regeneration) =====
# in addition to gen_quality, compute the TailGAN-paper
# strategy PnL (BuyHold + EigenPort static(50) + MR + TF) + empirical VaR/ES relative error.
# gan_data windows are auto-built below via build_windows() (skipped if present);
# TransMat is auto-created by configure_from_project and thresholds are computed on first use.
import sys as _sys
_TG_DIR = os.path.join(LOCAL_BASE_DIRECTORY, "modules", "TailGAN")
if _TG_DIR not in _sys.path:
    _sys.path.insert(0, _TG_DIR)

# build the TailGAN rolling-window CSVs (data/TailGAN/gan_data/indices2_6d) from data/indices.csv
# if missing; needed by the TailGAN-style VaR/ES metric. Imported from modules/TailGAN/run.py.
import run as _tg_run   # import-safe: run.py's pipeline is __main__-guarded
_gan_dir = os.path.join(_tg_run.YOUR_PATH, "gan_data", _tg_run.DATA_NAME)
if glob.glob(os.path.join(_gan_dir, "*.csv")):
    print(f"[TailGAN] gan_data exists -> skip build_windows()")
else:
    print(f"[TailGAN] building rolling-window CSVs -> {_gan_dir}")
    _tg_run.build_windows()

import tailgan_eval

_TG = tailgan_eval.configure_from_project(LOCAL_BASE_DIRECTORY)
TG_ALPHAS = (0.05, 0.025, 0.01)


def aggregate_tailgan_vares_ablation(tag_label_map, out_name="ablation__tailgan_vares", n_runs=N_RUNS):
    """aggregate ported VaR/ES relative error over seeds 0..n-1 as mean+/-std (real = test 6d = real_data)."""
    cols = {}
    for tag, label in tag_label_map.items():
        rp = file_manager.get_result_path(tag)
        per_seed = []
        for seed in range(n_runs):
            f = os.path.join(rp, f"{seed}.npy")
            if not os.path.exists(f):
                continue
            fake = np.load(f).astype(np.float32)
            per_seed.append(tailgan_eval.vares_relerror(_TG.Compute_PNL, _TG.Tensor, real_data, fake, TG_ALPHAS))
        if not per_seed:
            print(f"  (skip {tag}: no npy)")
            continue
        keys = list(per_seed[0].keys())
        arr = np.array([[d[k] for k in keys] for d in per_seed])
        m, sd = arr.mean(0), arr.std(0)
        cols[label] = pd.Series([_pm(a, b) for a, b in zip(m, sd)], index=keys)
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


print("[Ablation summary - TailGAN-style VaR/ES (ported)]")
summary_ablation_tailgan_vares = aggregate_tailgan_vares_ablation(tag_label_map)
print(summary_ablation_tailgan_vares)

[TailGAN] gan_data exists -> skip build_windows()
Namespace(n_epochs=3000, batch_size=1000, lr_D=1e-07, lr_G=1e-06, temp=0.01, b1=0.5, b2=0.999, latent_dim=1000, len=50000, n_rows=5, n_cols=100, n_critic_G=1, n_critic_D=1, static_way='LShort', strategies=['Port', 'MR', 'TF'], n_trans=50, Cap=10, WH=10, ratios=[1.0, 1.0], thresholds_pct=[[31, 69]], data_name='1_Gauss+1_AR50+1_AR-12+1_GARCH-T5+1_GARCH-T10', tickers=['Gauss', 'AR50', 'AR-12', 'GARCH-T5', 'GARCH-T10'], noise_name='t5', alphas=[0.05], W=10.0, score='quant', numNN=10, project=True, version='Test1')
[Ablation summary - TailGAN-style VaR/ES (ported)]
saved: c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/statistics/ablation__tailgan_vares.csv
                      Baseline   w/o RV Decomp.     w/o LambertW  \
RE_VaR(0.05)   0.3815 ± 0.0062  0.6578 ± 0.0055  0.4271 ± 0.0048   
RE_ES(0.05)    0.3507 ± 0.0040  0.6338 ± 0.0045  0.3956 ± 0.0044   
RE_VaR(0.025)  0.3525 ± 0.0082  0.6372 ± 0.0061  0.3969 ± 0.0053   
RE_ES(0